# 01 — Exploratory Data Analysis (EDA)
## easyMoney | TFM Data Science & AI — Nuclio School

**Objetivo:** Construir un nuevo autoservicio de Business Intelligence para el equipo, 
que permita responder preguntas clave como:
- ¿Cuántos productos hemos vendido este mes?
- ¿Son los clientes nuevos o los existentes quienes más contratan?
- ¿Cuál es el perfil demográfico de nuestros clientes por producto?

La entrega incluirá un dashboard interactivo y una presentación para el Comité de Dirección.

**Datasets:**
- `commercial_activity_df` — customer status, entry channel, segment (5.96M rows x 17 monthly snapshots)
- `products_df` — binary flags for 15 financial products per customer per snapshot
- `sociodemographic_df` — age, gender, salary, region, country

---

In [20]:
import pandas as pd
import os
import glob
from pathlib import Path

# ── Setup visualizaciones ──────────────────────────────────────────
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Paleta de colores easyMoney
EM_GREEN  = '#4CAF50'
EM_DARK   = '#2E7D32'
EM_LIGHT  = '#A5D6A7'
EM_GRAY   = '#78909C'
EM_ORANGE = '#FF9800'

print("✓ Plotly listo para visualizaciones!")
print(f"Versión plotly: {px.__version__ if hasattr(px, '__version__') else 'ok'}")

✓ Plotly listo para visualizaciones!
Versión plotly: ok


## 1. Carga de Datos

In [21]:
DATA_DIR = Path('../../data/raw')   

df_commercial = pd.read_csv(DATA_DIR / 'commercial_activity_df.csv', index_col=0)
df_products   = pd.read_csv(DATA_DIR / 'products_df.csv',            index_col=0)
df_socio      = pd.read_csv(DATA_DIR / 'sociodemographic_df.csv',    index_col=0)

for name, df in [('commercial', df_commercial), ('products', df_products), ('socio', df_socio)]:
    print(f'{name:15s} -> {df.shape[0]:>9,} rows  x  {df.shape[1]:>2} cols')

print('\nData loaded successfully')

commercial      -> 5,962,924 rows  x   6 cols
products        -> 5,962,924 rows  x  17 cols
socio           -> 5,962,924 rows  x   8 cols

Data loaded successfully


Los datos han sido proporcionados por el equipo de IT (Frank) como un volcado de la 
base de datos del autoservicio de BI existente, estructurado en **3 tablas separadas** 
que comparten las claves `pk_cid` (identificador de cliente) y `pk_partition` (fecha 
de ingesta, equivalente al período de análisis).


| Tabla | Filas | Columnas | Contenido |
|---|---|---|---|
| `commercial_activity` | 5,962,924 | 6 | Actividad comercial del cliente |
| `products` | 5,962,924 | 17 | Productos contratados (flags 0/1) |
| `sociodemographic` | 5,962,924 | 8 | Perfil sociodemográfico del cliente |

## 2. Exploración de la Estructura

Antes de cualquier análisis, inspeccionamos los tipos de datos, las primeras filas 
y la distribución de valores en cada tabla para entender con qué estamos trabajando.


In [22]:
# ── Paso 2: Vista rápida de cada tabla ─────────────────────────────────────

for name, df in [('commercial', df_commercial), 
                ('products',   df_products), 
                ('socio',      df_socio)]:
    print(f"\n{'='*50}")
    print(f"  {name.upper()}")
    print(f"{'='*50}")
    print(df.dtypes)
    print("\nPrimeras 3 filas:")
    print(df.head(3))


  COMMERCIAL
pk_cid               int64
pk_partition        object
entry_date          object
entry_channel       object
active_customer    float64
segment             object
dtype: object

Primeras 3 filas:
    pk_cid pk_partition  entry_date entry_channel  active_customer  \
0  1375586   2018-01-28  2018-01-12           KHL              1.0   
1  1050611   2018-01-28  2015-08-10           KHE              0.0   
2  1050612   2018-01-28  2015-08-10           KHE              0.0   

              segment  
0   02 - PARTICULARES  
1  03 - UNIVERSITARIO  
2  03 - UNIVERSITARIO  

  PRODUCTS
pk_cid                  int64
pk_partition           object
short_term_deposit      int64
loans                   int64
mortgage                int64
funds                   int64
securities              int64
long_term_deposit       int64
em_account_pp           int64
credit_card             int64
payroll               float64
pension_plan          float64
payroll_account         int64
emc_account 

### Hallazgos principales:

**Tabla commercial:**
- `pk_partition` y `entry_date` están almacenados como `object` → necesitan conversión a fecha
- `active_customer` solo tiene **2 valores únicos** (0/1) → flag binario correcto
- `segment` tiene **3 valores únicos** → segmentación comercial simple
- `entry_channel` tiene **68 valores únicos** → alta cardinalidad, investigar

**Tabla products:**
- Todos los productos son **flags binarios (0/1)** → estructura limpia
- `payroll` y `pension_plan` almacenados como `float64` → deben ser enteros
- `em_account_pp` tiene **1 solo valor único** → columna sin información, eliminar

**Tabla socio:**
- `region_code` almacenado como `float64` → debe convertirse a string (es un código)
- `gender` tiene **2 valores únicos** → binario correcto
- `age` tiene **104 valores únicos** → rango de edades razonable
- `salary` tiene **258,629 valores únicos** → variable continua, confirma que es ingreso real

---

## 3. Análisis de Calidad de Datos (Nulos)

Revisamos el porcentaje de valores faltantes en cada tabla para decidir 
la estrategia de tratamiento antes de construir ningún análisis.

In [23]:
# ── Paso 3: Nulos y valores únicos ─────────────────────────────────────────

for name, df in [('commercial', df_commercial), 
                 ('products',   df_products), 
                 ('socio',      df_socio)]:
    print(f"\n{'='*50}")
    print(f"  {name.upper()} — Nulos (%)")
    print(f"{'='*50}")
    null_pct = (df.isnull().sum() / len(df) * 100).round(2)
    print(null_pct[null_pct > 0] if null_pct.any() else "  Sin nulos")
    
    print(f"\n  Valores únicos por columna:")
    for col in df.columns:
        print(f"  {col:<25} {df[col].nunique():>8} únicos")


  COMMERCIAL — Nulos (%)
entry_channel    2.23
segment          2.25
dtype: float64

  Valores únicos por columna:
  pk_cid                      456373 únicos
  pk_partition                    17 únicos
  entry_date                    1499 únicos
  entry_channel                   68 únicos
  active_customer                  2 únicos
  segment                          3 únicos

  PRODUCTS — Nulos (%)
  Sin nulos

  Valores únicos por columna:
  pk_cid                      456373 únicos
  pk_partition                    17 únicos
  short_term_deposit               2 únicos
  loans                            2 únicos
  mortgage                         2 únicos
  funds                            2 únicos
  securities                       2 únicos
  long_term_deposit                2 únicos
  em_account_pp                    1 únicos
  credit_card                      2 únicos
  payroll                          2 únicos
  pension_plan                     2 únicos
  payroll_account        

| Tabla | Campo | % Nulos | Estrategia |
|---|---|---|---|
| commercial | `entry_channel` | 2.23% | Rellenar con 'UNKNOWN' |
| commercial | `segment` | 2.25% | Imputación inteligente por salary + region_code + age |
| socio | `region_code` | 0.04% | Negligible, rellenar con 'UNKNOWN' |
| socio | `salary` | 25.36% | Imputación inteligente por grupo de edad |
| products | — | 0% | Sin nulos ✓ |

> **Decisión clave sobre salary:** Con un 25% de nulos no podemos eliminar filas.
> Tras analizar la distribución por edad de los valores nulos, detectamos que el patrón
> no es aleatorio. La mayoría corresponde a jóvenes de 18-24 años (posibles becarios
> o estudiantes) y menores de 18. Por ello aplicamos una **imputación por grupo de edad**:
>
> | Grupo | Edad | Criterio | Valor imputado |
> |---|---|---|---|
> | Menores | < 18 | Sin ingresos reales | 0 |
> | Jóvenes | 18-24 | Becarios/estudiantes | Mediana del grupo (€88,496) |
> | Activos | 25-64 | Trabajadores en activo | Mediana del grupo (€87,861) |
> | Jubilados | 65+ | Pensionistas | Mediana del grupo (€102,809) |
>
> Esta estrategia es más realista que una mediana global porque respeta el perfil
> económico real de cada etapa vital. Se añade el flag `salary_imputed` para
> trazabilidad completa.

---

## 4. Limpieza y Conversión de Tipos

In [24]:
# ── Diagnóstico: fechas problemáticas en entry_date ────────────────────────

# Primero convertimos pk_partition (que funciona bien)
df_commercial['pk_partition'] = pd.to_datetime(df_commercial['pk_partition'])
df_products['pk_partition']   = pd.to_datetime(df_products['pk_partition'])
df_socio['pk_partition']      = pd.to_datetime(df_socio['pk_partition'])

# Ver qué hay en entry_date alrededor de la posición 688
print("Muestra de entry_date problemáticas:")
bad_dates = pd.to_datetime(df_commercial['entry_date'], errors='coerce')
mask = bad_dates.isna() & df_commercial['entry_date'].notna()
print(f"Total fechas inválidas: {mask.sum()}")
print(df_commercial[mask]['entry_date'].value_counts().head(20))

Muestra de entry_date problemáticas:
Total fechas inválidas: 6413
entry_date
2019-02-29    4621
2015-02-29    1792
Name: count, dtype: int64


In [25]:
# ── Fix: corregir fechas inválidas en entry_date ───────────────────────────

# Reemplazar fechas imposibles antes de convertir
df_commercial['entry_date'] = df_commercial['entry_date'].replace({
    '2019-02-29': '2019-02-28',
    '2015-02-29': '2015-02-28'
})

# Ahora sí convertir a datetime
df_commercial['entry_date'] = pd.to_datetime(df_commercial['entry_date'])

# Verificar que no quedan nulos inesperados
print(f"Nulos en entry_date tras fix: {df_commercial['entry_date'].isna().sum()}")
print(f"Tipo: {df_commercial['entry_date'].dtype}")
print(f"\nRango de fechas:")
print(f"  Más antigua: {df_commercial['entry_date'].min()}")
print(f"  Más reciente: {df_commercial['entry_date'].max()}")

Nulos en entry_date tras fix: 0
Tipo: datetime64[ns]

Rango de fechas:
  Más antigua: 2015-01-01 00:00:00
  Más reciente: 2019-05-31 00:00:00


In [26]:
# ── Fix: convertir floats con posibles NaN a int en products ───────────────

# Primero verificar qué hay exactamente
print("Nulos reales en payroll:", df_products['payroll'].isna().sum())
print("Nulos reales en pension_plan:", df_products['pension_plan'].isna().sum())

# Rellenar cualquier NaN con 0 antes de convertir a int
df_products['payroll']      = df_products['payroll'].fillna(0).astype(int)
df_products['pension_plan'] = df_products['pension_plan'].fillna(0).astype(int)

# Verificar
print(f"\npayroll dtype: {df_products['payroll'].dtype}")
print(f"pension_plan dtype: {df_products['pension_plan'].dtype}")
print(f"Valores únicos payroll: {df_products['payroll'].unique()}")
print(f"Valores únicos pension_plan: {df_products['pension_plan'].unique()}")

Nulos reales en payroll: 61
Nulos reales en pension_plan: 61

payroll dtype: int64
pension_plan dtype: int64
Valores únicos payroll: [0 1]
Valores únicos pension_plan: [0 1]


In [27]:
# ── Paso 4: Limpieza y conversión de tipos ─────────────────────────────────

# 4.1 Convertir fechas
df_commercial['pk_partition'] = pd.to_datetime(df_commercial['pk_partition'])
df_products['pk_partition']   = pd.to_datetime(df_products['pk_partition'])
df_socio['pk_partition']      = pd.to_datetime(df_socio['pk_partition'])

# 4.2 Convertir floats que deberían ser int en products
float_cols_products = ['payroll', 'pension_plan']
df_products[float_cols_products] = df_products[float_cols_products].astype(int)

# 4.3 region_code: float → string (es un código, no un número)
df_socio['region_code'] = df_socio['region_code'].fillna(-1).astype(int).astype(str)
df_socio['region_code'] = df_socio['region_code'].replace('-1', None)


# 4.4 Salary: imputación inteligente por grupo de edad
def impute_salary_final(df):
    df = df.copy()
    df['salary_imputed'] = df['salary'].isna()
    
    # Menores de 18 → 0
    mask_minor = (df['salary'].isna()) & (df['age'] < 18)
    df.loc[mask_minor, 'salary'] = 0
    
    # 18-24 → mediana del grupo
    median_young = df[(df['salary'].notna()) & 
                      (df['age'].between(18, 24))]['salary'].median()
    mask_young = (df['salary'].isna()) & (df['age'].between(18, 24))
    df.loc[mask_young, 'salary'] = median_young
    
    # 25-64 → mediana del grupo
    median_working = df[(df['salary'].notna()) & 
                        (df['age'].between(25, 64))]['salary'].median()
    mask_working = (df['salary'].isna()) & (df['age'].between(25, 64))
    df.loc[mask_working, 'salary'] = median_working
    
    # 65+ → mediana del grupo
    median_retired = df[(df['salary'].notna()) & 
                        (df['age'] >= 65)]['salary'].median()
    mask_retired = (df['salary'].isna()) & (df['age'] >= 65)
    df.loc[mask_retired, 'salary'] = median_retired
    
    return df

df_socio = impute_salary_final(df_socio)
print(f"✓ Nulos restantes: {df_socio['salary'].isna().sum()}")


# 4.5 Eliminar em_account_pp (un solo valor único → sin información)
print(f"em_account_pp valores únicos: {df_products['em_account_pp'].unique()}")
df_products = df_products.drop(columns=['em_account_pp'])

# 4.6 entry_channel → moda por segmento
df_commercial['entry_channel'] = df_commercial['entry_channel'].fillna(
    df_commercial.groupby('segment')['entry_channel'].transform(
        lambda x: x.mode()[0] if not x.mode().empty else 'UNKNOWN'
    )
)
df_commercial['entry_channel'] = df_commercial['entry_channel'].fillna('UNKNOWN')

# 4.7 segment → imputación inteligente por salary + region_code + age
# Primero necesitamos unir con socio para tener age y region_code
segment_to_num = {'03 - UNIVERSITARIO': 1, '02 - PARTICULARES': 2, '01 - TOP': 3}
num_to_segment = {v: k for k, v in segment_to_num.items()}

df_commercial['segment_num'] = df_commercial['segment'].map(segment_to_num)

# Merge temporal para imputar
temp = df_commercial.merge(
    df_socio[['pk_cid','pk_partition','salary','region_code','age']], 
    on=['pk_cid','pk_partition'], how='left'
)

group_median = temp.groupby(['salary','region_code','age'])['segment_num'].transform('median')
df_commercial['segment_num'] = df_commercial['segment_num'].fillna(group_median)

# Fallback global
df_commercial['segment_num'] = df_commercial['segment_num'].fillna(
    df_commercial['segment_num'].median()
)

# Convertir de vuelta a texto
df_commercial['segment'] = df_commercial['segment_num'].round().astype(int).map(num_to_segment)
df_commercial = df_commercial.drop(columns=['segment_num'])

print(f"✓ Nulos en segment: {df_commercial['segment'].isna().sum()}")
print(f"✓ Distribución segment:")
print(df_commercial['segment'].value_counts())
# ── Verificación ───────────────────────────────────────────────────────────
print("\nTipos después de limpieza:")
for name, df in [('commercial', df_commercial), 
                 ('products',   df_products), 
                 ('socio',      df_socio)]:
    print(f"\n{name}:")
    print(df.dtypes)

print("\nNulos restantes:")
for name, df in [('commercial', df_commercial), 
                 ('products',   df_products), 
                 ('socio',      df_socio)]:
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    print(f"\n{name}: {nulls.to_dict() if len(nulls) else 'Sin nulos'}")

✓ Nulos restantes: 0
em_account_pp valores únicos: [0]
✓ Nulos en segment: 0
✓ Distribución segment:
segment
03 - UNIVERSITARIO    4026906
02 - PARTICULARES     1837902
01 - TOP                98116
Name: count, dtype: int64

Tipos después de limpieza:

commercial:
pk_cid                      int64
pk_partition       datetime64[ns]
entry_date         datetime64[ns]
entry_channel              object
active_customer           float64
segment                    object
dtype: object

products:
pk_cid                         int64
pk_partition          datetime64[ns]
short_term_deposit             int64
loans                          int64
mortgage                       int64
funds                          int64
securities                     int64
long_term_deposit              int64
credit_card                    int64
payroll                        int64
pension_plan                   int64
payroll_account                int64
emc_account                    int64
debit_card              

In [28]:
# ── Fix final de nulos restantes ───────────────────────────────────────────

# Gender: 25 nulos → UNKNOWN
df_socio['gender'] = df_socio['gender'].fillna('UNKNOWN')

# Region_code: mantener None pero rellenar con 'UNKNOWN' para análisis
df_socio['region_code'] = df_socio['region_code'].fillna('UNKNOWN')

# ── Verificación final limpia ──────────────────────────────────────────────
print("\n✓ Nulos finales:")
for name, df in [('commercial', df_commercial), 
                 ('products',   df_products), 
                 ('socio',      df_socio)]:
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    print(f"  {name}: {nulls.to_dict() if len(nulls) else 'Sin nulos ✓'}")

print("✓ Limpieza completada!")


✓ Nulos finales:
  commercial: Sin nulos ✓
  products: Sin nulos ✓
  socio: Sin nulos ✓
✓ Limpieza completada!


| Problema | Solución aplicada |
|---|---|
| `entry_date` con fechas imposibles (29/02 en años no bisiestos) | Reemplazadas por 28/02 del mismo año — 6,413 registros corregidos |
| `payroll` y `pension_plan` como float64 con 61 NaNs ocultos | Rellenados con 0 y convertidos a int64 |
| `em_account_pp` con un único valor (0) | Columna eliminada — no aporta información |
| `region_code` como float64 | Convertido a string categórico |
| `salary` con 25.36% de nulos | Imputación inteligente por grupo de edad: <18 → 0, 18-24 → €88,496, 25-64 → €87,861, 65+ → €102,809 |
| `gender` con 25 nulos | Rellenado con 'UNKNOWN' |
| `segment` con 2.25% nulos | Imputación inteligente por salary + region_code + age → 67.5% UNIVERSITARIO, 30.8% PARTICULARES, 1.6% TOP |
| `entry_channel` con 2.23% nulos | Imputado con moda por segmento + fallback UNKNOWN |

> **Estado final:** Las 3 tablas están completamente limpias y listas para 
> el Feature Engineering. Se añadió el flag `salary_imputed` para trazabilidad.

---

## 5. Feature Engineering

In [29]:
# ── Paso 5: Feature Engineering ────────────────────────────────────────────

# 5.1 Unir las 3 tablas en un único dataframe maestro
df = df_commercial.merge(df_products, on=['pk_cid','pk_partition'], how='inner') \
                  .merge(df_socio,    on=['pk_cid','pk_partition'], how='inner')

print(f"✓ Tabla maestra: {df.shape[0]:,} filas × {df.shape[1]} columnas")

# 5.2 Definir columnas de productos
product_cols = ['short_term_deposit','loans','mortgage','funds','securities',
                'long_term_deposit','credit_card','payroll','pension_plan',
                'payroll_account','emc_account','debit_card','em_account_p',
                'em_acount']

print(f"✓ Productos: {len(product_cols)} columnas")

# 5.3 Total de productos por cliente por período
df['total_products'] = df[product_cols].sum(axis=1)
print(f"\nDistribución de productos por cliente:")
print(df['total_products'].value_counts().sort_index())

✓ Tabla maestra: 5,962,924 filas × 27 columnas
✓ Productos: 14 columnas

Distribución de productos por cliente:
total_products
0    1121507
1    3995714
2     528593
3     150269
4     105720
5      42890
6      14809
7       2799
8        573
9         50
Name: count, dtype: int64


In [30]:
# ── Paso 5 (continuación): más features ────────────────────────────────────

# 5.4 Identificar clientes nuevos vs existentes por partición
# Un cliente es "nuevo" en la partición en que aparece por primera vez
first_partition = df.groupby('pk_cid')['pk_partition'].min().reset_index()
first_partition.columns = ['pk_cid', 'first_partition']

df = df.merge(first_partition, on='pk_cid', how='left')
df['is_new_client'] = (df['pk_partition'] == df['first_partition']).astype(int)

print(f"Clientes nuevos por período:")
print(df.groupby('pk_partition')['is_new_client'].sum().to_string())

# 5.5 Delta de productos: productos nuevos contratados respecto al período anterior
df = df.sort_values(['pk_cid','pk_partition'])

for col in product_cols:
    df[f'{col}_prev'] = df.groupby('pk_cid')[col].shift(1)

prev_cols = [f'{col}_prev' for col in product_cols]
df['total_products_prev'] = df[prev_cols].sum(axis=1)

# Nuevas contrataciones = tenía 0 antes y ahora tiene 1
df['new_contracts'] = 0
for col in product_cols:
    df['new_contracts'] += ((df[f'{col}_prev'] == 0) & (df[col] == 1)).astype(int)

# Limpiar columnas temporales
df = df.drop(columns=prev_cols)
if 'total_products_prev' in df.columns:
    df = df.drop(columns=['total_products_prev'])

print(f"\nNuevas contrataciones por período:")
print(df.groupby('pk_partition')['new_contracts'].sum().to_string())

# 5.6 Antigüedad del cliente en meses
df['client_age_months'] = ((df['pk_partition'] - df['entry_date']) / 
                            pd.Timedelta(days=30)).round().astype(int)

print(f"\nAntigüedad media del cliente: {df['client_age_months'].mean():.1f} meses")
print(f"Antigüedad máxima: {df['client_age_months'].max()} meses")

Clientes nuevos por período:
pk_partition
2018-01-28    239493
2018-02-28      3767
2018-03-28      3411
2018-04-28      2952
2018-05-28      3031
2018-06-28      2845
2018-07-28     83840
2018-08-28     14596
2018-09-28     23353
2018-10-28     28205
2018-11-28     15557
2018-12-28      7402
2019-01-28      6926
2019-02-28      6193
2019-03-28      5690
2019-04-28      4581
2019-05-28      4531

Nuevas contrataciones por período:
pk_partition
2018-01-28        0
2018-02-28    11165
2018-03-28    11597
2018-04-28    10591
2018-05-28    10444
2018-06-28    14536
2018-07-28    13262
2018-08-28    14400
2018-09-28    15384
2018-10-28    18608
2018-11-28    16835
2018-12-28    19655
2019-01-28    13994
2019-02-28    20856
2019-03-28    16794
2019-04-28    15447
2019-05-28    17009

Antigüedad media del cliente: 20.9 meses
Antigüedad máxima: 54 meses


In [31]:
# ── Paso 5 (continuación): features finales ────────────────────────────────

# 5.7 Categorías de edad
df['age_group'] = pd.cut(df['age'], 
                        bins=[0, 18, 25, 35, 45, 55, 65, 100],
                        labels=['<18', '18-24', '25-35', '35-45', '45-55', '55-65', '65+'])     

# 5.8 Categorías de salario
df['salary_group'] = pd.cut(df['salary'],
                        bins=[-1, 0, 20000, 40000, 60000, 80000, 120000, 999999],
                        labels=['sin_ingreso', '<20k', '20-40k', '40-60k', '60-80k', '80-120k', '120k+'])       

# 5.9 Resumen final de la tabla maestra
print("✓ Feature Engineering completado!")
print(f"\nTabla maestra final: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"\nColumnas disponibles:")
for col in df.columns:
    print(f"  {col}")

print(f"\nEjemplo — distribución por grupo de edad:")
print(df['age_group'].value_counts().sort_index())

print(f"\nEjemplo — distribución por grupo de salario:")
print(df['salary_group'].value_counts().sort_index())

✓ Feature Engineering completado!

Tabla maestra final: 5,962,924 filas × 34 columnas

Columnas disponibles:
  pk_cid
  pk_partition
  entry_date
  entry_channel
  active_customer
  segment
  short_term_deposit
  loans
  mortgage
  funds
  securities
  long_term_deposit
  credit_card
  payroll
  pension_plan
  payroll_account
  emc_account
  debit_card
  em_account_p
  em_acount
  country_id
  region_code
  gender
  age
  deceased
  salary
  salary_imputed
  total_products
  first_partition
  is_new_client
  new_contracts
  client_age_months
  age_group
  salary_group

Ejemplo — distribución por grupo de edad:
age_group
<18        37716
18-24    3236391
25-35    1330555
35-45     705526
45-55     361264
55-65     170512
65+       120759
Name: count, dtype: int64

Ejemplo — distribución por grupo de salario:
salary_group
sin_ingreso       6872
<20k             22209
20-40k          286783
40-60k          736109
60-80k          854807
80-120k        2713918
120k+          1333749
Name: c

### Resultados — Feature Engineering

A partir de las 3 tablas limpias se construyó una **tabla maestra única** 
de 5,962,924 filas × 34 columnas con las siguientes features adicionales:

| Feature | Descripción |
|---|---|
| `total_products` | Número total de productos contratados por cliente en cada período |
| `is_new_client` | Flag: 1 si es la primera vez que el cliente aparece en los datos |
| `new_contracts` | Número de productos nuevos contratados respecto al período anterior |
| `client_age_months` | Antigüedad del cliente en meses desde su primera contratación |
| `first_partition` | Fecha de la primera aparición del cliente en los datos |
| `age_group` | Grupos de edad: <18, 18-24, 25-35, 35-45, 45-55, 55-65, 65+ |
| `salary_group` | Grupos de salario: sin_ingreso, <20k, 20-40k, 40-60k, 60-80k, 80-120k, 120k+ |

### Insights preliminares del Feature Engineering:

- La base de clientes es predominantemente **joven (<25 años)** — perfil 
  típico de una plataforma fintech digital. Destaca el grupo **<18 años** 
  con salary imputado a 0 (sin ingresos) y el grupo **18-24** como posibles 
  becarios o estudiantes, lo que explica el alto porcentaje de nulos en salary.
- El nivel salarial es **sorprendentemente alto** — el grupo 80-120k es 
  el más numeroso, lo que sugiere un perfil de cliente con capacidad 
  de inversión elevada
- La mayoría de clientes tienen **0 o 1 productos** — enorme oportunidad 
  de cross-sell alineada con la estrategia de penetración de Ansoff
- Las nuevas contrataciones muestran una **tendencia creciente** 
  (~10,000 → ~20,000/mes) a lo largo de los 17 períodos
- El spike de **83,840 clientes nuevos en julio 2018** requiere 
  investigación — posible campaña de captación masiva o migración de datos

---

## 6. Visualizaciones y Dashboard BI

In [32]:
# ── Gráfico 1: Evolución de clientes activos y nuevas contrataciones ────────

# Agregar datos por período
period_summary = df.groupby('pk_partition').agg(
    total_clients        = ('pk_cid', 'count'),
    new_clients          = ('is_new_client', 'sum'),
    total_new_contracts  = ('new_contracts', 'sum'),
    active_clients       = ('active_customer', 'sum')
).reset_index()

period_summary['pk_partition'] = period_summary['pk_partition'].astype(str).str[:10]

# Gráfico con doble eje Y
fig1 = make_subplots(specs=[[{"secondary_y": True}]])

fig1.add_trace(go.Bar(
    x=period_summary['pk_partition'],
    y=period_summary['new_clients'],
    name='Clientes nuevos',
    marker_color=EM_LIGHT,
    opacity=0.7
), secondary_y=False)

fig1.add_trace(go.Scatter(
    x=period_summary['pk_partition'],
    y=period_summary['total_clients'],
    name='Total clientes',
    line=dict(color=EM_DARK, width=3),
    mode='lines+markers'
), secondary_y=False)

fig1.add_trace(go.Scatter(
    x=period_summary['pk_partition'],
    y=period_summary['total_new_contracts'],
    name='Nuevas contrataciones',
    line=dict(color=EM_ORANGE, width=2, dash='dot'),
    mode='lines+markers'
), secondary_y=True)

fig1.update_layout(
    title='Evolución de clientes y nuevas contrataciones por período',
    xaxis_title='Período',
    legend=dict(orientation='h', yanchor='bottom', y=1.02),
    plot_bgcolor='white',
    height=450
)
fig1.update_yaxes(title_text='Nº clientes', secondary_y=False)
fig1.update_yaxes(title_text='Nuevas contrataciones', secondary_y=True)
fig1.update_xaxes(tickangle=45)

fig1.show()
print("✓ Gráfico 1 generado!")

✓ Gráfico 1 generado!


Una cosa que no tiene sentido:

La primera barra (enero de 2018) muestra ~240k “nuevos clientes”, lo cual es técnicamente correcto, pero visualmente resulta engañoso: no representa un crecimiento real, sino simplemente la primera fotografía de todos los clientes ya existentes. Esto va a confundir a Carol y al comité.

In [33]:
# ── Gráfico 1 corregido: excluir primera partición del bar ─────────────────

first_period = period_summary['pk_partition'].min()
period_plot = period_summary.copy()
period_plot.loc[period_plot['pk_partition'] == first_period, 'new_clients'] = 0

fig1 = make_subplots(specs=[[{"secondary_y": True}]])

fig1.add_trace(go.Bar(
    x=period_plot['pk_partition'],
    y=period_plot['new_clients'],
    name='Clientes nuevos',
    marker_color=EM_LIGHT,
    opacity=0.7
), secondary_y=False)

fig1.add_trace(go.Scatter(
    x=period_plot['pk_partition'],
    y=period_plot['total_clients'],
    name='Total clientes',
    line=dict(color=EM_DARK, width=3),
    mode='lines+markers'
), secondary_y=False)

fig1.add_trace(go.Scatter(
    x=period_plot['pk_partition'],
    y=period_plot['total_new_contracts'],
    name='Nuevas contrataciones',
    line=dict(color=EM_ORANGE, width=2, dash='dot'),
    mode='lines+markers'
), secondary_y=True)

fig1.update_layout(
    title='Evolución de clientes y nuevas contrataciones por período',
    xaxis_title='Período',
    legend=dict(orientation='h', yanchor='bottom', y=1.02),
    plot_bgcolor='white',
    height=450
)
fig1.update_yaxes(title_text='Nº clientes', secondary_y=False)
fig1.update_yaxes(title_text='Nuevas contrataciones', secondary_y=True)
fig1.update_xaxes(tickangle=45)

fig1.show()
print("✓ Gráfico 1 corregido!")

✓ Gráfico 1 corregido!


El gráfico cuenta una historia de negocio clara:

- Crecimiento sostenido de clientes: de 240k a 440k en 17 meses 
- El pico de julio de 2018 sigue siendo visible en la barra de nuevos clientes; conviene destacárselo a Carol
- Los nuevos contratos muestran una tendencia al alza: de ~11k a ~20k al mes, lo que refleja un impulso saludable de cross-sell 
- Ahora las barras son honestas: solo muestran clientes realmente nuevos en cada periodo 

In [34]:
# ── Gráfico 2: Tasa de penetración por producto ────────────────────────────

# Usar solo la última partición para foto actual
last_partition = df['pk_partition'].max()
df_last = df[df['pk_partition'] == last_partition]

total_clients_last = len(df_last)

penetration = pd.DataFrame({
    'producto': product_cols,
    'clientes': [df_last[col].sum() for col in product_cols],
})
penetration['penetracion_pct'] = (penetration['clientes'] / total_clients_last * 100).round(2)
penetration = penetration.sort_values('penetracion_pct', ascending=True)

# Etiquetas más legibles
label_map = {
    'em_acount': 'Cuenta easyMoney',
    'payroll': 'Domiciliaciones',
    'em_account_p': 'Cuenta easyMoney+',
    'debit_card': 'Tarjeta débito',
    'credit_card': 'Tarjeta crédito',
    'payroll_account': 'Cuenta nómina',
    'emc_account': 'Cuenta Crypto',
    'short_term_deposit': 'Depósito C/P',
    'long_term_deposit': 'Depósito L/P',
    'pension_plan': 'Plan pensiones',
    'funds': 'Fondos inversión',
    'securities': 'Valores',
    'mortgage': 'Hipoteca',
    'loans': 'Préstamos',
    'em_account_pp': 'Cuenta easyMoney++'
}
penetration['label'] = penetration['producto'].map(label_map)

fig2 = go.Figure(go.Bar(
    x=penetration['penetracion_pct'],
    y=penetration['label'],
    orientation='h',
    marker_color=EM_GREEN,
    text=penetration['penetracion_pct'].apply(lambda x: f'{x:.1f}%'),
    textposition='outside'
))

fig2.update_layout(
    title=f'Tasa de penetración por producto — {str(last_partition)[:10]}',
    xaxis_title='% clientes con este producto',
    yaxis_title='',
    plot_bgcolor='white',
    height=500,
    xaxis=dict(range=[0, penetration['penetracion_pct'].max() * 1.2])
)

fig2.show()
print("✓ Gráfico 2 generado!")

✓ Gráfico 2 generado!


Lo que revela este gráfico:

- La Cuenta easyMoney domina con un 66,9%: es el producto de entrada, como era de esperar, ya que fue su primer producto.
- Todo lo demás está por debajo del 10%, lo que evidencia una gran brecha de cross-sell.
- Préstamos, Hipoteca, Cuenta easyMoney+, Depósito C/P aparecen todos con un 0,0%: estos productos son prácticamente inexistentes dentro de la cartera.
- La Tarjeta débito (9,8%) es el segundo producto más popular, lo cual tiene sentido como complemento natural de la cuenta principal.
- La Cuenta Crypto (5,6%) resulta sorprendentemente alta para ser un producto de nicho.
- Esto respalda directamente la estrategia de penetración de mercado de Ansoff: la base actual de clientes está claramente infraatendida.

In [35]:
# ── Gráfico 3: Contrataciones nuevos vs existentes por período ─────────────

contracts_by_type = df[df['pk_partition'] != df['pk_partition'].min()].groupby(
    ['pk_partition', 'is_new_client']
)['new_contracts'].sum().reset_index()

contracts_by_type['pk_partition'] = contracts_by_type['pk_partition'].astype(str).str[:10]
contracts_by_type['tipo_cliente'] = contracts_by_type['is_new_client'].map({
    0: 'Cliente existente',
    1: 'Cliente nuevo'
})

fig3 = px.bar(
    contracts_by_type,
    x='pk_partition',
    y='new_contracts',
    color='tipo_cliente',
    color_discrete_map={
        'Cliente existente': EM_DARK,
        'Cliente nuevo': EM_LIGHT
    },
    title='Nuevas contrataciones: clientes nuevos vs existentes por período',
    labels={
        'pk_partition': 'Período',
        'new_contracts': 'Nuevas contrataciones',
        'tipo_cliente': 'Tipo de cliente'
    },
    barmode='stack'
)

fig3.update_layout(
    plot_bgcolor='white',
    height=450,
    legend=dict(orientation='h', yanchor='bottom', y=1.02),
    xaxis_tickangle=45
)

fig3.show()
print("✓ Gráfico 3 generado!")

✓ Gráfico 3 generado!


“Cliente nuevo” = 0 en todos los periodos. Esto significa que nuestra variable new_contracts solo captó contratos procedentes de clientes ya existentes. La razón es lógica: cuando un cliente aparece por primera vez (is_new_client = 1), no existe un periodo anterior con el que comparar, por lo que el delta siempre es 0.

En realidad, esto responde a una lógica de negocio correcta: el primer producto de un cliente nuevo no se considera un “nuevo contrato” en sentido de cross-sell.

Sin embargo, nos falta una parte importante de la foto: qué aportan los clientes nuevos en su día 1

In [36]:
# ── Fix: contratos totales nuevos vs existentes (incluyendo primer producto) 

# Para clientes nuevos → contar sus productos en su primera partición
new_client_products = df[df['is_new_client'] == 1].copy()
new_client_products['contracts_new_client'] = new_client_products[product_cols].sum(axis=1)

new_per_period = new_client_products.groupby('pk_partition').agg(
    contracts_new_clients=('contracts_new_client', 'sum'),
    new_client_count=('pk_cid', 'count')
).reset_index()

# Para clientes existentes → usar new_contracts (delta)
existing_per_period = df[df['is_new_client'] == 0].groupby('pk_partition').agg(
    contracts_existing_clients=('new_contracts', 'sum'),
    existing_client_count=('pk_cid', 'count')
).reset_index()

# Unir
contracts_full = new_per_period.merge(existing_per_period, on='pk_partition', how='outer').fillna(0)
contracts_full['pk_partition'] = contracts_full['pk_partition'].astype(str).str[:10]

# Excluir primera partición
contracts_full = contracts_full[contracts_full['pk_partition'] != '2018-01-28']

print("Contratos por tipo - visión completa:")
contracts_full[['pk_partition','contracts_new_clients',
                'contracts_existing_clients']].to_string()

# Calcular porcentajes
contracts_full['total'] = (contracts_full['contracts_new_clients'] + 
                           contracts_full['contracts_existing_clients'])
contracts_full['pct_new'] = (contracts_full['contracts_new_clients'] / 
                              contracts_full['total'] * 100).round(1)
contracts_full['pct_existing'] = (contracts_full['contracts_existing_clients'] / 
                                   contracts_full['total'] * 100).round(1)

print(contracts_full[['pk_partition','contracts_new_clients',
                       'contracts_existing_clients','pct_new']].to_string())

# Gráfico
fig3c = go.Figure()

fig3c.add_trace(go.Bar(
    x=contracts_full['pk_partition'],
    y=contracts_full['contracts_existing_clients'],
    name='Cliente existente',
    marker_color=EM_DARK
))

fig3c.add_trace(go.Bar(
    x=contracts_full['pk_partition'],
    y=contracts_full['contracts_new_clients'],
    name='Cliente nuevo',
    marker_color=EM_ORANGE
))

fig3c.update_layout(
    title='Contrataciones totales: clientes nuevos vs existentes por período',
    xaxis_title='Período',
    yaxis_title='Contrataciones',
    barmode='stack',
    plot_bgcolor='white',
    height=450,
    legend=dict(orientation='h', yanchor='bottom', y=1.02),
    xaxis_tickangle=45
)

fig3c.show()
print("✓ Gráfico 3 definitivo!")

Contratos por tipo - visión completa:
   pk_partition  contracts_new_clients  contracts_existing_clients  pct_new
1    2018-02-28                   4017                     11165.0     26.5
2    2018-03-28                   3684                     11597.0     24.1
3    2018-04-28                   3150                     10591.0     22.9
4    2018-05-28                   3177                     10444.0     23.3
5    2018-06-28                   3052                     14536.0     17.4
6    2018-07-28                  12789                     13262.0     49.1
7    2018-08-28                  11097                     14400.0     43.5
8    2018-09-28                  16166                     15384.0     51.2
9    2018-10-28                  16317                     18608.0     46.7
10   2018-11-28                   8552                     16835.0     33.7
11   2018-12-28                   3307                     19655.0     14.4
12   2019-01-28                   3834            

✓ Gráfico 3 definitivo!


Lo que nos dicen los datos:

- A comienzos de 2018 (febrero-junio), aproximadamente el 75% de las contrataciones procedían de clientes existentes y alrededor del 25% de clientes nuevos, lo que refleja una base de cross-sell saludable.

- Entre julio y octubre de 2018, la contribución de los clientes nuevos sube hasta cerca del 50%, lo que confirma el impacto de la gran campaña de captación.

- La tendencia en 2019 muestra que la aportación de los clientes nuevos cae con fuerza hasta situarse entre el 9% y el 14% en mayo de 2019, lo que indica que la base está madurando y que los clientes existentes dominan cada vez más la contratación.

- El cambio estratégico que quiere Carol ya se está produciendo de forma natural: los clientes existentes son cada vez más la principal fuente de nuevos contratos.

Esta es la respuesta a la pregunta de Carol: “¿Son los clientes nuevos quienes contratan o los que ya teníamos?” → Los clientes que ya teníamos, y además la tendencia se está acelerando.

In [37]:
# ── Gráfico 4: Distribución de productos por cliente (cross-sell opportunity)

product_dist = df[df['pk_partition'] == last_partition]['total_products'].value_counts().sort_index().reset_index()
product_dist.columns = ['num_productos', 'clientes']
product_dist['pct'] = (product_dist['clientes'] / product_dist['clientes'].sum() * 100).round(1)

fig4 = go.Figure(go.Bar(
    x=product_dist['num_productos'].astype(str),
    y=product_dist['clientes'],
    marker_color=[EM_ORANGE if x <= 1 else EM_GREEN for x in product_dist['num_productos']],
    text=product_dist['pct'].apply(lambda x: f'{x}%'),
    textposition='outside'
))

fig4.update_layout(
    title='Distribución de productos por cliente — oportunidad de cross-sell (May 2019)',
    xaxis_title='Número de productos contratados',
    yaxis_title='Número de clientes',
    plot_bgcolor='white',
    height=420,
    annotations=[dict(
        x=0.5, y=1.08, xref='paper', yref='paper',
        text='<b>Naranja = clientes con 0-1 productos (objetivo cross-sell)</b>',
        showarrow=False, font=dict(size=11, color=EM_ORANGE)
    )]
)
fig4.show()

# ── Gráfico 5: Perfil demográfico — edad y salario ─────────────────────────

fig5 = make_subplots(rows=1, cols=2, 
                      subplot_titles=('Distribución por grupo de edad',
                                     'Distribución por grupo de salario'))

age_dist = df[df['pk_partition'] == last_partition]['age_group'].value_counts().sort_index()
salary_dist = df[df['pk_partition'] == last_partition]['salary_group'].value_counts().sort_index()

fig5.add_trace(go.Bar(
    x=age_dist.index.astype(str),
    y=age_dist.values,
    marker_color=EM_GREEN,
    name='Edad'
), row=1, col=1)

fig5.add_trace(go.Bar(
    x=salary_dist.index.astype(str),
    y=salary_dist.values,
    marker_color=EM_DARK,
    name='Salario'
), row=1, col=2)

fig5.update_layout(
    title='Perfil demográfico de clientes — Mayo 2019',
    plot_bgcolor='white',
    height=400,
    showlegend=False
)
fig5.update_xaxes(tickangle=45)
fig5.show()

print("✓ Gráficos 4 y 5 generados!")

✓ Gráficos 4 y 5 generados!


Gráfico 4 Oportunidad de cross-sell:

- El 25,1% de los clientes no tiene ningún producto: están en la base de datos, pero no han contratado nada.

- El 60,6% tiene solo 1 producto, muy probablemente únicamente la Cuenta easyMoney.

- En total, el 85,7% de los clientes tiene 0 o 1 productos: esta es precisamente la oportunidad de penetración de mercado de Ansoff de la que hablaba Carol.

- Solo el 0,2% tiene 6 o más productos, algo extremadamente poco frecuente.

Gráfico 5 Perfil demográfico:

- El segmento de menores de 25 años domina claramente, lo que indica una base muy joven y digital-native.

- La distribución salarial es bimodal, con picos en 80–120k y 120k+: esto sugiere clientes de ingresos altos a pesar de su juventud, algo poco habitual y que conviene destacar.

- Hay muy pocos clientes de ingresos bajos, lo que confirma que no se trata de un producto orientado al mercado masivo.

### Conclusiones para el Comité de Dirección

> La estrategia de **penetración de mercado (Ansoff)** está plenamente justificada 
> por los datos:
> 1. El **85.7% de los clientes tienen 0 o 1 productos** — la oportunidad de 
>    cross-sell es enorme
> 2. Los **clientes existentes ya generan el 86-91% de las nuevas contrataciones** 
>    en 2019 — la base está receptiva
> 3. El perfil demográfico (joven, alto poder adquisitivo) es **ideal para productos 
>    de inversión y ahorro**
> 4. Productos como Préstamos, Hipoteca y Fondos de inversión tienen penetración 
>    casi nula — **quick wins potenciales** para la estrategia comercial

---

In [38]:
# ── Guardar tabla maestra para reutilizar en tareas posteriores ────────────

df.to_parquet('../../data/processed/master_df.parquet', index=False)
print(f"✓ Tabla maestra guardada: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"  Ruta: ../../data/processed/master_df.parquet")

✓ Tabla maestra guardada: 5,962,924 filas × 34 columnas
  Ruta: ../../data/processed/master_df.parquet


## 7. Conclusiones y KPIs — Resumen Ejecutivo

### Respuestas a las preguntas de Carol

| Pregunta | Respuesta |
|---|---|
| ¿Cuántos clientes activos tenemos? | 331,588 (74.9% de la base) |
| ¿Cuántos productos por cliente? | Media de 1.32 productos/cliente activo |
| ¿Quién contrata más, nuevos o existentes? | Existentes — 88% de los contratos en 2019 |
| ¿Cuál es el producto más contratado? | Cuenta easyMoney (66.9%) |
| ¿Cuántos clientes tienen solo 1 producto? | 268,286 — el 60.6% de la base |

### KPIs de seguimiento definidos

| KPI | Valor actual | Objetivo |
|---|---|---|
| Tasa de cross-sell (2+ productos) | 14.3% | > 20% |
| Penetración Tarjeta débito | 9.8% | > 20% |
| % contratos clientes existentes | ~88% | > 80% ✓ |
| Media productos/cliente activo | 1.32 | > 2.0 |
| Clientes con 0 productos | 25.1% | Reducción progresiva |

> **Conclusión estratégica:** El 85.7% de los clientes tiene 0 o 1 productos.
> La oportunidad de cross-sell es enorme y está alineada con la estrategia
> de penetración de Ansoff que propone Carol.

---

> **Siguiente paso →** `02_data_quality.ipynb` — Análisis de calidad de datos
> 
> **Dashboard →** `03_power_bi.ipynb` — Exportación y visualización en Power BI

## 8. Nota sobre Calidad de Datos

Tras el EDA inicial, se realizó un análisis exhaustivo de calidad de datos
en `02_data_quality.ipynb`. Se identificaron y documentaron las siguientes
anomalías mediante flags en la tabla maestra:

| Flag | Registros | % | Descripción |
|---|---|---|---|
| `age_anomaly` | 4,806 | 0.08% | Edad < 5, > 100, o menor de 18 con pension_plan |
| `deceased_anomaly` | 724 | 0.01% | Fallecido pero activo o con nuevas contrataciones |
| `entry_date_anomaly` | 10,780 | 0.18% | Entry_date 1-3 días posterior a pk_partition |

```

> **Siguiente paso →** `03_power_bi.ipynb` — Exportación de datos y dashboard BI
